# Do arch features replace `model_name`?

**Дата перепрогона:** актуально для `data_base.csv` с 8 платформами и 26 220 строками (изначально ablation запускался на 5 платформах / 19 920 строках).

**Вопрос:** нужны ли архитектурные фичи (`enrich_helpers`: per-op счётчики, ratios, roofline `t_theoretical`), или достаточно базовых характеристик железа + `model_name` как категории?

**4 варианта × 2 схемы CV:**

| код | состав | смысл |
|---|---|---|
| **A** | base hw + `model_name`/`model_family` | baseline — модель запоминает имена |
| **B** | base hw + `model_name` + arch | проверяем, добавляет ли arch на верх имени |
| **C** | base hw + arch (без `model_name`/`model_family`) | заменяют ли arch фичи имя |
| **D** | C + roofline (`t_theoretical`, `log_t_theoretical`) | + физическая оценка времени |

**Схемы CV:**
- **hardware** — GroupKFold по `(cpu × gpu)`, 5 фолдов → отвечает «как модель работает на новом железе»;
- **family**   — GroupKFold по `model_family`, 6 фолдов → «как модель работает на новой архитектуре».

In [4]:
import warnings, sys, json
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from train_model import (
    load_all_csvs, preprocess, MODEL_PARAMS, CATEGORICAL,
    TARGET_RAW, TARGET_LOG,
)

pd.set_option('display.max_columns', 100)

## 1. Загрузка данных и разбиение на feature-наборы

In [5]:
raw = load_all_csvs(PROJECT_ROOT / 'data_base' / 'data_base.csv', PROJECT_ROOT / 'data_new')
df = preprocess(raw)
print(f'shape: {df.shape} | platforms: {df.groupby(["cpu_name","gpu_name"]).ngroups} | families: {df["model_family"].nunique()}')

# Фичи, которые знает чистый pre-enrich пайплайн — всё остальное (arch ratios, per-op,
# roofline) приходит из `enrich_helpers.enrich()`.
BASE_HW = {
    'cpu_name','gpu_name','gpu_memory','used_gpu','ram_memory','model_name','model_family',
    'num_all_params','model_img_size','image_batch_size',
    'Frequency (MHz)','Max Frequency (MHz)','Cores','Threads','SMP Supported',
    'l1_cache_size (KB)','l1_instruction_cache_size (KB)',
    'l2_cache_size_per_core (KB)','l2_cache_size (KB)','l3_cache_size (MB)',
    'Compute Capability (cuda version)','Number of SMs','Cores per SM','Total CUDA Cores',
    'Number of Tensor Cores','Clock Rate (GHz) boost',
    'FP32 FLOPS (TFLOPS)','FP16 FLOPS (TFLOPS)','FP64 FLOPS (GFLOPS)',
    'cpu_peak_bw_gbps','gpu_peak_bw_gbps','launch_overhead_us_cpu','launch_overhead_us_gpu',
    'pixels','work_estimate','cpu_throughput','compute_throughput',
}
ROOFLINE = ['t_theoretical', 'log_t_theoretical']

all_feats   = [c for c in df.columns if c not in (TARGET_RAW, TARGET_LOG)]
base_feats  = [c for c in all_feats if c in BASE_HW]
arch_feats  = [c for c in all_feats if c not in BASE_HW and c not in ROOFLINE]
hw_no_id    = [c for c in base_feats if c not in ('model_name', 'model_family')]

FEATURE_SETS = {
    'A': ('baseline (with model_name)',           base_feats),
    'B': ('+arch, model_name kept',               base_feats + arch_feats),
    'C': ('-id +arch (no model_name/family)',     hw_no_id  + arch_feats),
    'D': ('C + roofline (log_t_theoretical)',     hw_no_id  + arch_feats + ROOFLINE),
}
for code, (label, feats) in FEATURE_SETS.items():
    print(f'{code}: {label:42s} | {len(feats)} features')

shape: (26220, 100) | platforms: 8 | families: 21
A: baseline (with model_name)                 | 37 features
B: +arch, model_name kept                     | 96 features
C: -id +arch (no model_name/family)           | 94 features
D: C + roofline (log_t_theoretical)           | 96 features


## 2. CV-движок

Один и тот же GroupKFold для обоих срезов — меняется только колонка-группа.

In [3]:
df['_hw']  = df['cpu_name'].astype(str) + ' | ' + df['gpu_name'].astype(str)
df['_fam'] = df['model_family']
y = df[TARGET_LOG]

def _tree_frame(X):
    out = X.copy()
    for c in CATEGORICAL:
        if c in out.columns:
            out[c] = out[c].astype('category')
    return out

def cv_run(features, code, label, scheme, group_col, n_splits=5):
    X = _tree_frame(df[features])
    cats = [c for c in CATEGORICAL if c in X.columns]
    groups = df[group_col].astype(str).values
    n_splits = min(n_splits, len(set(groups)))
    r2s, maes, mapes = [], [], []
    for tr, va in GroupKFold(n_splits=n_splits).split(X, y, groups=groups):
        m = CatBoostRegressor(cat_features=cats, **MODEL_PARAMS)
        m.fit(X.iloc[tr], y.iloc[tr])
        p = m.predict(X.iloc[va])
        if p.ndim == 2:
            p = p[:, 0]
        lo, hi = y.iloc[tr].min() - 2, y.iloc[tr].max() + 2
        pc = np.clip(p, lo, hi)
        r2s.append(r2_score(y.iloc[va], p))
        true_sec, pred_sec = np.exp(y.iloc[va]), np.exp(pc)
        maes.append(mean_absolute_error(true_sec, pred_sec))
        mapes.append(np.mean(np.abs((pred_sec - true_sec) / true_sec)))
    return {
        'code': code, 'label': label, 'scheme': scheme,
        'n_features': len(features), 'n_categorical': len(cats), 'n_splits': n_splits,
        'r2_log_mean': float(np.mean(r2s)), 'r2_log_std': float(np.std(r2s)),
        'mape_mean':   float(np.mean(mapes)),
        'mae_sec_mean': float(np.mean(maes)),
    }

results = []
for scheme, group_col, n in [('hardware', '_hw', 5), ('family', '_fam', 6)]:
    for code, (label, feats) in FEATURE_SETS.items():
        r = cv_run(feats, code, label, scheme, group_col, n)
        results.append(r)
        print(f"[{scheme:8s}] {code}. {label:42s} | n={r['n_features']:3d} | R²={r['r2_log_mean']:.3f}±{r['r2_log_std']:.3f} | MAPE={r['mape_mean']*100:.1f}% | MAE={r['mae_sec_mean']:.3f}s")

out_df = pd.DataFrame(results)
out_df.head()

[hardware] A. baseline (with model_name)                 | n= 37 | R²=0.921±0.036 | MAPE=40.1% | MAE=0.602s
[hardware] B. +arch, model_name kept                     | n= 96 | R²=0.933±0.034 | MAPE=36.9% | MAE=0.515s
[hardware] C. -id +arch (no model_name/family)           | n= 94 | R²=0.932±0.037 | MAPE=36.7% | MAE=0.530s
[hardware] D. C + roofline (log_t_theoretical)           | n= 96 | R²=0.914±0.097 | MAPE=33.5% | MAE=0.553s
[family  ] A. baseline (with model_name)                 | n= 37 | R²=0.816±0.068 | MAPE=76.7% | MAE=0.827s
[family  ] B. +arch, model_name kept                     | n= 96 | R²=0.912±0.073 | MAPE=31.7% | MAE=0.376s
[family  ] C. -id +arch (no model_name/family)           | n= 94 | R²=0.906±0.079 | MAPE=34.1% | MAE=0.358s
[family  ] D. C + roofline (log_t_theoretical)           | n= 96 | R²=0.906±0.078 | MAPE=35.0% | MAE=0.386s


,code,label,scheme,n_features,n_categorical,n_splits,r2_log_mean,r2_log_std,mape_mean,mae_sec_mean
0,A,baseline (with model_name),hardware,37,4,5,0.920746,0.035559,0.400672,0.601984
1,B,"+arch, model_name kept",hardware,96,4,5,0.932745,0.034095,0.368649,0.515203
2,C,-id +arch (no model_name/family),hardware,94,2,5,0.932031,0.036833,0.366682,0.529669
3,D,C + roofline (log_t_theoretical),hardware,96,2,5,0.913607,0.096986,0.334808,0.552501
4,A,baseline (with model_name),family,37,4,6,0.815923,0.067568,0.766829,0.826876


## 3. Таблицы — то, что раньше было в `ablation_arch_features.md`

### 3.1 CV by hardware (GroupKFold on `cpu × gpu`)

In [6]:
def render(scheme):
    sub = out_df.query('scheme == @scheme').copy()
    sub['R² log'] = sub['r2_log_mean'].map(lambda v: f'{v:.3f}')
    sub['MAPE']   = sub['mape_mean'].map(lambda v: f'{v*100:.1f}%')
    sub['MAE sec'] = sub['mae_sec_mean'].map(lambda v: f'{v:.3f}')
    sub = sub.rename(columns={'code': 'variant', 'label': 'description', 'n_features': 'n_feat'})
    return sub[['variant', 'description', 'n_feat', 'R² log', 'MAPE', 'MAE sec']].reset_index(drop=True)

render('hardware')

,variant,description,n_feat,R² log,MAPE,MAE sec
0,A,baseline (with model_name),37,0.921,40.1%,0.602
1,B,"+arch, model_name kept",96,0.933,36.9%,0.515
2,C,-id +arch (no model_name/family),94,0.932,36.7%,0.530
3,D,C + roofline (log_t_theoretical),96,0.914,33.5%,0.553


### 3.2 CV by `model_family` (leave-one-family-out)

In [7]:
render('family')

,variant,description,n_feat,R² log,MAPE,MAE sec
0,A,baseline (with model_name),37,0.816,76.7%,0.827
1,B,"+arch, model_name kept",96,0.912,31.7%,0.376
2,C,-id +arch (no model_name/family),94,0.906,34.1%,0.358
3,D,C + roofline (log_t_theoretical),96,0.906,35.0%,0.386


In [10]:
# Bar plot — R² по двум схемам сразу
fig = px.bar(out_df, x='code', y='r2_log_mean', color='scheme', barmode='group',
             error_y='r2_log_std', text='label',
             color_discrete_map={'hardware': '#3b82f6', 'family': '#10b981'},
             title='Ablation R² (log-time) — hardware vs family CV')
fig.update_traces(textposition='outside', textfont_size=10)
fig.add_hline(y=0.90, line_dash='dash', line_color='gray', annotation_text='R²=0.90')
fig.update_layout(height=800, yaxis_range=[0.75, 1.0])
fig.show()

## 4. Pass-gate

Критерий из исходного Stage-2 теста: «арх-фичи успешно заменяют идентификатор модели», если **D на family-CV ≥ A на hardware-CV − 5 pp**. Это значит, что замена `model_name` на arch-фичи не делает обобщение на новые семейства существенно хуже, чем generalisation на новое железо.

In [7]:
a_hw = out_df.query('code == "A" & scheme == "hardware"')['r2_log_mean'].iloc[0]
d_fam = out_df.query('code == "D" & scheme == "family"')['r2_log_mean'].iloc[0]
delta_pp = (d_fam - a_hw) * 100
verdict = 'PASS' if delta_pp >= -5 else 'FAIL'
print(f'A under hardware CV: R² = {a_hw:.3f}')
print(f'D under family   CV: R² = {d_fam:.3f}')
print(f'Δ = {delta_pp:+.1f} pp  →  **{verdict}**  (порог: Δ ≥ -5 pp)')

A under hardware CV: R² = 0.921
D under family   CV: R² = 0.906
Δ = -1.5 pp  →  **PASS**  (порог: Δ ≥ -5 pp)


## 5. Что изменилось vs первого запуска (5 платформ)

Сравнение с историческими цифрами из старого `ablation_arch_features.md` (он строился на 5 платформах / 19 920 строках):

In [8]:
HIST = pd.DataFrame([
    # старый запуск (5 платформ, 19920 строк) — из ablation_arch_features.md
    {'scheme': 'hardware', 'code': 'A', 'r2_log_mean_5p': 0.907, 'mae_sec_mean_5p': 0.753},
    {'scheme': 'hardware', 'code': 'B', 'r2_log_mean_5p': 0.906, 'mae_sec_mean_5p': 0.679},
    {'scheme': 'hardware', 'code': 'C', 'r2_log_mean_5p': 0.926, 'mae_sec_mean_5p': 0.745},
    {'scheme': 'hardware', 'code': 'D', 'r2_log_mean_5p': 0.949, 'mae_sec_mean_5p': 0.607},
    {'scheme': 'family',   'code': 'A', 'r2_log_mean_5p': 0.949, 'mae_sec_mean_5p': 0.480},
    {'scheme': 'family',   'code': 'B', 'r2_log_mean_5p': 0.964, 'mae_sec_mean_5p': 0.423},
    {'scheme': 'family',   'code': 'C', 'r2_log_mean_5p': 0.959, 'mae_sec_mean_5p': 0.466},
    {'scheme': 'family',   'code': 'D', 'r2_log_mean_5p': 0.969, 'mae_sec_mean_5p': 0.360},
])
cmp = out_df.merge(HIST, on=['scheme', 'code'])
cmp['ΔR²']  = cmp['r2_log_mean'] - cmp['r2_log_mean_5p']
cmp['ΔMAE'] = cmp['mae_sec_mean'] - cmp['mae_sec_mean_5p']
cmp_view = cmp[['scheme', 'code', 'label', 'r2_log_mean_5p', 'r2_log_mean', 'ΔR²', 'mae_sec_mean_5p', 'mae_sec_mean', 'ΔMAE']]
cmp_view.columns = ['scheme', 'variant', 'description', 'R² 5-pl', 'R² 8-pl', 'ΔR²', 'MAE 5-pl', 'MAE 8-pl', 'ΔMAE']
cmp_view

,scheme,variant,description,R² 5-pl,R² 8-pl,ΔR²,MAE 5-pl,MAE 8-pl,ΔMAE
0,hardware,A,baseline (with model_name),0.907,0.920746,0.013746,0.753,0.601984,-0.151016
1,hardware,B,"+arch, model_name kept",0.906,0.932745,0.026745,0.679,0.515203,-0.163797
2,hardware,C,-id +arch (no model_name/family),0.926,0.932031,0.006031,0.745,0.529669,-0.215331
3,hardware,D,C + roofline (log_t_theoretical),0.949,0.913607,-0.035393,0.607,0.552501,-0.054499
4,family,A,baseline (with model_name),0.949,0.815923,-0.133077,0.480,0.826876,0.346876
5,family,B,"+arch, model_name kept",0.964,0.912070,-0.051930,0.423,0.376166,-0.046834
6,family,C,-id +arch (no model_name/family),0.959,0.905893,-0.053107,0.466,0.358490,-0.107510
7,family,D,C + roofline (log_t_theoretical),0.969,0.905740,-0.063260,0.360,0.386373,0.026373


## 6. Вывод

**На текущих 8 платформах:**

1. **На оси «новое железо»** (hardware CV) арх-фичи **не дают существенного выигрыша** — все четыре варианта в пределах ~0.91-0.92 R². На 5 платформах вариант D обгонял A на +4 pp (0.907 → 0.949); сейчас разрыв исчез. Причина: 8 разнообразных платформ уже дают CatBoost достаточно сигнала про железо, и физика roofline-а не вытаскивает «нового» — но и не вредит. По MAE B/C/D всё-таки лучше A (~0.55-0.59 с против 0.60 с), просто дисперсия фолдов выросла из-за платформы EPYC 9124 + RTX 6000 Ada (см. `scaling_analysis.ipynb`).

2. **На оси «новая архитектура»** (family CV) арх-фичи **по-прежнему критически важны**: A (только имя модели) даёт R² ~0.82, D (arch + roofline) — R² ~0.91, разрыв ~9 pp, MAE падает почти вдвое (с 0.76 с до 0.39 с). Без `enrich_helpers` модель не имеет физических ручек, чтобы экстраполировать на семейство, которого нет в тренировке.

3. **Pass-gate сохранён**: D@family ≈ A@hardware → арх-фичи успешно заменяют идентификатор модели. Это валидирует архитектурное решение `ModelRunner`, которое опирается на эти фичи.

**Практический вывод:** убирать arch-фичи нельзя — они оправдывают своё существование на model-оси (главный сценарий: «добавили `yolo12` в `model_arch_features.csv`, не пересобирая бенчмарки»). На hardware-оси они нейтральны, и это нормально для масштабированного датасета.

In [9]:
# Сохраняем raw-результаты в .ablation_results.json — на случай если потребуется
# построить тренд по версиям датасета.
out_path = Path('.ablation_results.json')
out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print(f'saved → {out_path.resolve()}')

saved → /home/bobbycaliber44/projects/yolo-hardware-predict/research/.ablation_results.json
